In [1]:
from ultralytics import YOLO
import os, glob, json, statistics

/home/arina_belova_jetbrains_com/three_od_models/yolov12/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = YOLO("yolov12n.pt")
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=2, bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, trac

In [3]:
RF100_ROOT = "/home/arina_belova_jetbrains_com/roboflow-100-benchmark/rf100"
yaml_files = sorted(glob.glob(f"{RF100_ROOT}/*/data.yaml"))
data_folders = sorted(glob.glob(f"{RF100_ROOT}/*/valid/images"))
print(f"Found {len(yaml_files)} yaml files and {len(data_folders)} data folders")

Found 99 yaml files and 99 data folders


In [4]:
results = {}

for yaml_path, data_folder in zip(yaml_files, data_folders):
    ds_name = os.path.basename(os.path.dirname(yaml_path))
    print(f"\n=== {ds_name} ===")
    try:
        model_weights_path = f"/home/arina_belova_jetbrains_com/rf100_runs/yolov12n_{ds_name}/weights/best.pt"

        if os.path.exists(model_weights_path):
            print(f"  ﷿﷿﷿ Found existing model at {model_weights_path}, skipping training")
            model = YOLO(model_weights_path)
        else:
            print(f"NEED TO TRAIN THE MODEL")
            # model = YOLO("yolov12n.pt")
            # model.train(
            #     data=yaml_path,
            #     epochs=50,
            #     imgsz=640,
            #     batch=16,
            #     device=0,
            #     project="rf100_runs",
            #     name=f"yolov12n_{ds_name}",
            #     exist_ok=True,
            #     verbose=False,
            # )

        print("Validating the model")
        m = model.val(data=yaml_path, split="val")
        #imgs = model.predict(source=data_folder, save=True)
        results[ds_name] = {"mAP50": float(m.box.map50), "mAP50-95": float(m.box.map)} #, "imgs": imgs}
        print(results)
        print(f"  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")
    except Exception as e:
        print(f"  ﷿﷿﷿ skipped {ds_name}: {e}")
        results[ds_name] = {"error": str(e)}

# Save
with open("./rf100_yolov12n_results.json", "w") as f:
    json.dump(results, f, indent=2)

# Aggregate
maps = [r["mAP50"] for r in results.values() if "mAP50" in r]
print(f"\nRF100 mean mAP@50 (YOLOv12n): {statistics.mean(maps):.4f} over {len(maps)} datasets")


=== 4-fold-defect ===
  ﷿﷿﷿ Found existing model at /home/arina_belova_jetbrains_com/rf100_runs/yolov12n_4-fold-defect/weights/best.pt, skipping training
Validating the model
Ultralytics 8.3.63 🚀 Python-3.11.15 torch-2.2.2+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81152MiB)
YOLOv12n summary (fused): 376 layers, 2,508,539 parameters, 0 gradients, 5.8 GFLOPs


val: Scanning /home/arina_belova_jetbrains_com/roboflow-100-benchmark/rf100/4-fold-defect/valid/labels.cache... 134 images, 0 backgrounds, 0 corrupt: 100%|██████████| 134/134 [00:00<?, ?it/s]
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2+cu121
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:08<00:00,  1.04it/s]


                   all        134       2277      0.958      0.932       0.96      0.529
Speed: 2.3ms preprocess, 20.2ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to /home/arina_belova_jetbrains_com/yolov12/runs/detect/val2
did we validate the model?
{'4-fold-defect': {'mAP50': 0.9598881549221485, 'mAP50-95': 0.5285234264124298}}
  mAP50=0.960  mAP50-95=0.529

=== abdomen-mri ===
  ﷿﷿﷿ Found existing model at /home/arina_belova_jetbrains_com/rf100_runs/yolov12n_abdomen-mri/weights/best.pt, skipping training
Validating the model
Ultralytics 8.3.63 🚀 Python-3.11.15 torch-2.2.2+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81152MiB)
YOLOv12n summary (fused): 376 layers, 2,508,539 parameters, 0 gradients, 5.8 GFLOPs


val: Scanning /home/arina_belova_jetbrains_com/roboflow-100-benchmark/rf100/abdomen-mri/valid/labels.cache... 479 images, 0 backgrounds, 0 corrupt: 100%|██████████| 479/479 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 30/30 [00:04<00:00,  6.09it/s]


                   all        479        484      0.949      0.936       0.96      0.566
Speed: 0.7ms preprocess, 2.5ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /home/arina_belova_jetbrains_com/yolov12/runs/detect/val3
did we validate the model?
{'4-fold-defect': {'mAP50': 0.9598881549221485, 'mAP50-95': 0.5285234264124298}, 'abdomen-mri': {'mAP50': 0.9602238919917359, 'mAP50-95': 0.5664093961709384}}
  mAP50=0.960  mAP50-95=0.566

=== acl-x-ray ===
  ﷿﷿﷿ Found existing model at /home/arina_belova_jetbrains_com/rf100_runs/yolov12n_acl-x-ray/weights/best.pt, skipping training
Validating the model
Ultralytics 8.3.63 🚀 Python-3.11.15 torch-2.2.2+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81152MiB)
YOLOv12n summary (fused): 376 layers, 2,508,539 parameters, 0 gradients, 5.8 GFLOPs


...

                   all        235       1605      0.634      0.779      0.765      0.488
                  crop         17         47      0.517      0.787      0.726       0.48
                  weed        228       1558      0.751       0.77      0.804      0.496
Speed: 1.3ms preprocess, 4.8ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /home/arina_belova_jetbrains_com/yolov12/runs/detect/val99
did we validate the model?
{'4-fold-defect': {'mAP50': 0.9598881549221485, 'mAP50-95': 0.5285234264124298}, 'abdomen-mri': {'mAP50': 0.9602238919917359, 'mAP50-95': 0.5664093961709384}, 'acl-x-ray': {'mAP50': 0.995, 'mAP50-95': 0.7485975132650964}, 'activity-diagrams-qdobr': {'mAP50': 0.48075664540258717, 'mAP50-95': 0.308541450935207}, 'aerial-cows': {'mAP50': 0.7376879401317721, 'mAP50-95': 0.37074828836875223}, 'aerial-pool': {'mAP50': 0.6849945053668992, 'mAP50-95': 0.3853558619964307}, 'aerial-spheres': {'mAP50': 0.9929954907337086, 'mAP50-95': 0.7863712815744828}

val: Scanning /home/arina_belova_jetbrains_com/roboflow-100-benchmark/rf100/wine-labels/valid/labels.cache... 841 images, 0 backgrounds, 0 corrupt: 100%|██████████| 841/841 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 53/53 [00:09<00:00,  5.76it/s]


                   all        841       4613      0.752      0.532      0.551      0.398
     AlcoholPercentage        240        244       0.57      0.574        0.5      0.322
Appellation AOC DOC AVARegion        587        648      0.569      0.595      0.601      0.422
Appellation QualityLevel        164        169       0.66      0.666      0.632      0.408
        CountryCountry        298        391      0.575      0.443      0.494      0.324
         Distinct Logo        714        879      0.822      0.869        0.9      0.761
  Established YearYear        140        152      0.646      0.414       0.47       0.29
            Maker-Name        828       1059      0.777      0.811      0.858      0.696
               Organic         12         15          1          0    0.00674    0.00448
           Sustainable         25         27          1          0     0.0102    0.00619
Sweetness-Brut-SecSweetness-Brut-Sec         54         55      0.862      0.455      0.544      0.37

val: Scanning /home/arina_belova_jetbrains_com/roboflow-100-benchmark/rf100/x-ray-rheumatology/valid/labels.cache... 34 images, 1 backgrounds, 0 corrupt: 100%|██████████| 34/34 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:04<00:00,  1.64s/it]


                   all         34        764      0.919      0.839      0.866       0.57
              artefact          1          1          1          0          0          0
      distal phalanges         33        179      0.967      0.974       0.97      0.544
 fifth metacarpal bone         33         36      0.931      0.972      0.974      0.672
 first metacarpal bone         32         35      0.943      0.939      0.946      0.725
fourth metacarpal bone         32         34      0.876      0.912      0.933       0.62
intermediate phalanges         33        144      0.967      0.986      0.979      0.621
    proximal phalanges         33        179      0.957      0.985      0.971      0.683
                radius         31         34      0.946          1      0.973      0.745
second metacarpal bone         33         37      0.951      0.946      0.963      0.686
soft tissue calcination         14         14      0.737      0.403      0.742      0.223
 third metacarpal bo

In [6]:
from ultralytics import YOLO
model = YOLO("/home/arina_belova_jetbrains_com/rf100_runs/yolov12n_4-fold-defect/weights/best.pt")
m = model.val(
    data="/home/arina_belova_jetbrains_com/roboflow-100-benchmark/rf100/4-fold-defect/data.yaml",
    split="test", workers=0, verbose=True,
)
print(m.box.map50, m.box.map)

Ultralytics 8.3.63 🚀 Python-3.11.15 torch-2.2.2+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81152MiB)
YOLOv12n summary (fused): 376 layers, 2,508,539 parameters, 0 gradients, 5.8 GFLOPs


val: Scanning /home/arina_belova_jetbrains_com/roboflow-100-benchmark/rf100/4-fold-defect/test/labels... 33 images, 0 backgrounds, 0 corrupt: 100%|██████████| 33/33 [00:00<00:00, 1366.60it/s]

val: New cache created: /home/arina_belova_jetbrains_com/roboflow-100-benchmark/rf100/4-fold-defect/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:02<00:00,  1.10it/s]


                   all         33        423      0.758      0.752      0.704      0.302
Speed: 2.6ms preprocess, 16.7ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /home/arina_belova_jetbrains_com/yolov12/runs/detect/val103
0.7044112458691113 0.30244315767640206


## Unified RF100 benchmark: baseline -> fine-tune -> val curve -> test

Fine-tuning lives in the standalone script **`finetune_rf100.py`** so it can run outside Jupyter. Per dataset (`cable-damage`, `bone-fracture-7fylg`, `soda-bottles`) it runs the same four steps and writes `../comparison_results/yolov12n_results.json`:

1. **Zero-shot baseline** validation (before fine-tuning)
2. **Fine-tune** for N epochs (default 10; set `--epochs N`)
3. **Per-epoch validation curve** (parsed from the run's `results.csv`)
4. **Final `test`-split** stats

**How to run the fine-tune** (from this directory, this venv):

    .venv/bin/python finetune_rf100.py --epochs 10

The cell below loads the JSON that script produced and shows the baseline vs test table plus the per-dataset validation curves. See `../compare_models.ipynb` for the cross-model comparison.

> **Baseline note:** the pretrained weights emit COCO class ids, while RF100 reuses those ids for different classes, so zero-shot mAP is ~0. It's the honest "before" reference, not a bug.

In [ ]:
# --- Unified benchmark: run fine-tune OUTSIDE the notebook (recommended) ---
# From this directory, with this venv:
#     .venv/bin/python finetune_rf100.py --epochs 10
# Re-run at any depth with --epochs N. To launch from inside Jupyter, uncomment:
# !.venv/bin/python finetune_rf100.py --epochs 10

import json
from pathlib import Path
import matplotlib.pyplot as plt

RESULTS_JSON = Path("../comparison_results/yolov12n_results.json")
data = json.loads(RESULTS_JSON.read_text())
print(f"model={data['model']}  epochs={data['epochs']}  imgsz={data['imgsz']}\n")

# Baseline (zero-shot) vs final test per dataset.
for ds, e in data["datasets"].items():
    b, t = e["baseline_val"], e["final_test"]
    print(f"{ds}")
    print(f"  baseline (zero-shot): mAP50={b['mAP50']:.4f}  mAP50-95={b['mAP50_95']:.4f}")
    print(f"  final test          : mAP50={t['mAP50']:.4f}  mAP50-95={t['mAP50_95']:.4f}")

# Validation curve during fine-tuning, one figure per dataset.
for ds, e in data["datasets"].items():
    curve = e.get("val_curve", [])
    if not curve:
        continue
    ep = [c["epoch"] for c in curve]
    plt.figure(figsize=(8, 5))
    for key in ("mAP50", "mAP50_95"):
        plt.plot(ep, [c.get(key) for c in curve], marker="o", label=key)
    plt.title(f"{data['model']} - {ds}: validation metrics vs epoch")
    plt.xlabel("epoch"); plt.ylabel("value"); plt.grid(True, alpha=0.3); plt.legend()
    plt.tight_layout(); plt.show()

## One-shot COCO val2017 check (pretrained, no fine-tune)

Sanity reference: evaluate the **pretrained** COCO weights on **COCO `val2017`** (the 5000-image standard split these models were trained on). Unlike the RF100 zero-shot baseline (≈0, because RF100 relabels the class ids), this should reproduce roughly the model's published COCO mAP.

> Uses `val2017`, not `test2017` — COCO's test set has no public labels, so mAP can't be computed on it. Data is prepared once by `../prepare_coco_val.py` (downloads only the val split). The result is stored in `../comparison_results/coco_baseline.json`.

In [2]:
# --- One-shot COCO val2017 sanity check (pretrained weights, NO fine-tune) ---
# All three models are COCO-pretrained, so this should land near their published
# COCO mAP (unlike the ~0 RF100 zero-shot baseline, whose classes don't match COCO).
# Also times batch=1 predict() (warmup + CUDA sync) the same way as the RF-DETR
# benchmark, so FPS is comparable across models.
#
# Prepare the shared COCO val data once (idempotent, ~1 GB; downloads val2017 only):
#     python3 ../prepare_coco_val.py
# To launch from inside Jupyter, uncomment:
# !python3 ../prepare_coco_val.py

import json, time, statistics
from pathlib import Path
import torch
from ultralytics import YOLO

COCO_ROOT = Path.home() / "datasets" / "coco"
COCO_YAML = COCO_ROOT / "coco-val.yaml"
WEIGHTS = "yolov12m.pt"  # pretrained COCO weights

model = YOLO(WEIGHTS)
m = model.val(data=str(COCO_YAML), split="val", verbose=False)
coco = {"mAP50": float(m.box.map50), "mAP50_95": float(m.box.map), "num_images": 5000}

# Batch=1 latency / FPS over all val images (matches the RF-DETR timing loop).
val_images = sorted((COCO_ROOT / "images" / "val2017").glob("*.jpg"))
cuda = torch.cuda.is_available()
for _ in range(3):
    model.predict(str(val_images[0]), verbose=False)
if cuda:
    torch.cuda.synchronize()
lat = []
for p in val_images:
    if cuda:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    model.predict(str(p), verbose=False)
    if cuda:
        torch.cuda.synchronize()
    lat.append(time.perf_counter() - t0)
mean_s = statistics.mean(lat)
coco.update({"latency_ms_mean": 1000 * mean_s,
             "latency_ms_median": 1000 * statistics.median(lat),
             "fps": 1.0 / mean_s})
print(f"{WEIGHTS} COCO val2017: mAP50={coco['mAP50']:.4f}  mAP50-95={coco['mAP50_95']:.4f}"
      f"  |  {coco['latency_ms_mean']:.1f} ms/img  ({coco['fps']:.1f} FPS)")

BASE = Path("../comparison_results/coco_baseline.json")
allres = json.loads(BASE.read_text()) if BASE.exists() else {}
allres[Path(WEIGHTS).stem] = coco
BASE.write_text(json.dumps(allres, indent=2))
print("saved ->", BASE.resolve())

Ultralytics 8.3.63 🚀 Python-3.11.15 torch-2.2.2+cu121 CUDA:0 (NVIDIA A100-SXM4-80GB, 81152MiB)
YOLOv12m summary (fused): 402 layers, 19,638,208 parameters, 0 gradients, 59.8 GFLOPs


val: Scanning /home/arina_belova_jetbrains_com/datasets/coco/labels/val2017.cache... 5000 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2+cu121
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 313/313 [00:38<00:00,  8.05it/s]


                   all       5000      36335      0.728      0.638      0.695      0.526
Speed: 0.1ms preprocess, 2.9ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /home/arina_belova_jetbrains_com/yolov12/runs/detect/val136
yolov12m.pt COCO val2017: mAP50=0.6955  mAP50-95=0.5259  |  69.7 ms/img  (14.3 FPS)
saved -> /home/arina_belova_jetbrains_com/three_od_models/comparison_results/coco_baseline.json


## Model size: parameter count

Number of trainable parameters, written to `../comparison_results/model_params.json` so
`compare_models.ipynb` can show it next to the accuracy numbers.

In [1]:
# --- Model size: parameter count ---
import json
from pathlib import Path
from ultralytics import YOLO

WEIGHTS = "yolov12m.pt"
n = sum(p.numel() for p in YOLO(WEIGHTS).model.parameters())
print(f"{WEIGHTS}: {n:,} params ({n/1e6:.2f} M)")

B = Path("../comparison_results/model_params.json")
d = json.loads(B.read_text()) if B.exists() else {}
d[Path(WEIGHTS).stem] = {"num_params": int(n)}
B.write_text(json.dumps(d, indent=2))
print("saved ->", B.resolve())

/home/arina_belova_jetbrains_com/three_od_models/yolov12/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


yolov12m.pt: 19,670,784 params (19.67 M)
saved -> /home/arina_belova_jetbrains_com/three_od_models/comparison_results/model_params.json
